# 0. Problem
## 1070. Product Sales Analysis III — Medium
For each product, return its earliest sale year as `first_year` and that row's `quantity` and `price`.
Official: https://leetcode.com/problems/product-sales-analysis-iii/

# 1. Setup

In [ ]:
import pandas as pd
sales_rows=[(1,100,2008,10,5000),(2,100,2009,12,5000),(7,200,2011,15,9000),(8,200,2012,20,8500)]
sales_pd=pd.DataFrame(sales_rows,columns=["sale_id","product_id","year","quantity","price"])
sales_pd

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
spark=SparkSession.builder.getOrCreate()
sales_spark=spark.createDataFrame(sales_rows,["sale_id","product_id","year","quantity","price"])
sales_spark.createOrReplaceTempView("Sales")

# 2. SQL Solution

In [ ]:
sql_result=spark.sql("""WITH first_year AS (SELECT product_id,MIN(year) AS first_year FROM Sales GROUP BY product_id) SELECT s.product_id,f.first_year,s.quantity,s.price FROM Sales s JOIN first_year f ON s.product_id=f.product_id AND s.year=f.first_year ORDER BY s.product_id""")
sql_result.show(truncate=False)

# 3. pandas Solution

In [ ]:
first_year=sales_pd.groupby("product_id")["year"].transform("min")
result_pd=(sales_pd.loc[sales_pd["year"].eq(first_year),["product_id","year","quantity","price"]].rename(columns={"year":"first_year"}).sort_values("product_id").reset_index(drop=True))
result_pd

# 4. PySpark Solution

In [ ]:
first_year_spark=sales_spark.groupBy("product_id").agg(F.min("year").alias("first_year"))
result_spark=(sales_spark.alias("s").join(first_year_spark.alias("f"),(F.col("s.product_id")==F.col("f.product_id"))&(F.col("s.year")==F.col("f.first_year")),"inner").select(F.col("s.product_id").alias("product_id"),F.col("f.first_year").alias("first_year"),F.col("s.quantity").alias("quantity"),F.col("s.price").alias("price")).orderBy("product_id"))
result_spark.show(truncate=False)

# 5. Pattern Mapping
| Concept | SQL | pandas | PySpark |
|---|---|---|---|
| earliest/group | `MIN()` | `.transform("min")` | `F.min()` |
| retrieve full row | aggregate + join-back | boolean filter | aggregate + join-back |

# 6. Muscle-Memory Round

พิมพ์ใหม่เองโดยไม่ copy คำตอบด้านบน

In [ ]:
# MUSCLE MEMORY — SQL
# Rebuild using temp view(s): Sales

In [ ]:
# MUSCLE MEMORY — PANDAS
# Rebuild using: sales_pd

In [ ]:
# MUSCLE MEMORY — PYSPARK
# Rebuild using: sales_spark